# Modelo 4. DINOv3 con imagen GRZ y redshift

**Inferencia de propiedades físicas de galaxias a partir de imágenes multibanda** · Instituto Tecnológico y de Estudios Superiores de Monterrey · Maestría en Inteligencia Artificial Aplicada · **Equipo 24** · Colaboración con el Instituto de Astronomía de la UNAM

| Integrante | Matrícula |
|---|---|
| José Florencio Maguey Peralta | A01796727 |
| Erick Alberto Bustos Cruz | A01378966 |
| Luz Copelia Minutti Pérez | A01796921 |

**Asesores:** Dr. Iván Reyes Amezcua (ITESM) · Dr. Héctor Hernández Toledo y Dr. José Antonio Vázquez Mata (IA-UNAM)

**Fase de CRISP-ML(Q):** 3, *Machine Learning Model Engineering* (Studer et al. 2021). **Avance 4:** modelos alternativos.

## 1. Planteamiento

Este cuaderno evalúa un **modelo fundacional de visión**, DINOv3 (Siméoni et al. 2025), como extractor de la imagen para estimar la masa estelar (log M\*), la tasa de formación estelar (log SFR) y el índice D4000, junto con el redshift de cada galaxia.

DINOv3 es el modelo de visión más reciente de Meta. Se entrenó de forma autosupervisada, es decir, sin etiquetas, con 1,689 millones de imágenes de internet. Su modelo principal tiene 7,000 millones de parámetros y se destiló en una familia de modelos más pequeños; aquí usamos los transformadores de visión (ViT) con parches de 16 píxeles:

| Modelo | Parámetros | Dimensión del vector |
|---|---|---|
| ViT-S/16 | 21 M | 384 |
| ViT-B/16 | 86 M | 768 |
| ViT-L/16 | 300 M | 1,024 |
| ViT-L/16, versión satelital (SAT-493M) | 300 M | 1,024 |

Un ViT divide la imagen en parches, convierte cada parche en un vector (*token*) y los relaciona entre sí con mecanismos de atención. Además de los tokens de los parches, produce un token de clase (CLS) que resume la imagen completa. La aportación principal de DINOv3, el *Gram anchoring*, mantiene de buena calidad los vectores de cada parche aun a alta resolución.

**Qué sabemos de antemano.** Los autores evalúan DINOv3 congelado, con un modelo ligero encima. En astronomía, la evidencia publicada indica que este tipo de modelos congelados quedan por debajo de una red ajustada con galaxias:

- DINOv2 congelado logró R² = 0.55 en la masa estelar, contra 0.82 de una red supervisada (Parker et al. 2025).
- Con imágenes de Legacy Survey, el desempeño de los modelos congelados crece con su tamaño, pero DINOv3 se comporta de forma anómala: sus versiones más grandes se parecen más a su maestro que a otros modelos (UniverseTBD 2026).

Por eso evaluamos DINOv3 en dos etapas:

1. **Representaciones congeladas** (sección 5). Calculamos una vez los vectores de cada tamaño de modelo y entrenamos encima las cabezas del protocolo común. Comparamos el tamaño del modelo, la forma de presentar la imagen (bandas estandarizadas o la imagen en color de Lupton, más parecida a las fotos con que se entrenó DINOv3), la forma de resumir los tokens y la resolución. Al final ajustamos los hiperparámetros de la cabeza.
2. **Ajuste fino** (sección 6) del mejor tamaño, con dos estrategias para adaptar pocos parámetros a 7,000 imágenes: abrir solo los últimos bloques del ViT o agregar adaptadores de bajo rango (LoRA; Hu et al. 2022).

La entrada es un recorte central de 288 × 288 píxeles (75.5″) de la imagen de 300 px, porque los ViT necesitan un lado múltiplo de 16; la mitad de la luz de una galaxia típica queda dentro de 22 px, así que el recorte conserva la galaxia completa.

### Protocolo común de comparación

Los seis modelos del avance se comparan con las mismas reglas, de modo que las diferencias se puedan atribuir al algoritmo y no a cambios en los datos o en la forma de medir:

| Elemento | Regla común |
|---|---|
| Muestra | Galaxias con QCFLAG = 0, etiquetas y redshift válidos, y sin bandas vacías en el recorte; misma partición de entrenamiento, validación y prueba |
| Imagen | Bandas g, r y z en flujo físico, con `asinh(f / s_b)` (s_b: percentil 75 de \|f\| en entrenamiento) y estandarización por banda con entrenamiento |
| Variables escalares | El redshift, estandarizado con entrenamiento, se une al vector de la imagen antes de las cabezas |
| Salidas | Tres cabezas en paralelo, `Linear(d + 1, 256) → ReLU → Dropout(0.1) → Linear(256, 1)`, sobre etiquetas estandarizadas |
| Aumentación | Giros de 90° y reflejos en entrenamiento; nada en validación y prueba |
| Criterio de selección | Promedio entre propiedades del MSE normalizado, (RMSE / σ_entrenamiento)²; se conserva la época con el menor valor en validación |
| Métricas | MAE, RMSE, sesgo y R² por propiedad, en su escala original, y el tiempo de entrenamiento |
| Semillas | 42, 7 y 2026; la semilla 42 representa al modelo en la tabla comparativa |
| Prueba | Se abre una sola vez, al final, con la elección registrada antes |

## 2. Parámetros

| Parámetro | Valor | Significado |
|---|---|---|
| `SEMILLAS` | 42, 7, 2026 | Semillas; la 42 representa al modelo |
| `EXTRACTORES` | ViT-S, ViT-B, ViT-L y ViT-L satelital | Tamaños de DINOv3 que se comparan congelados |
| `PRUEBAS_CABEZA` | 40 | Configuraciones de la cabeza en la búsqueda sobre representaciones congeladas |
| `PRUEBAS_AJUSTE` | 16 | Configuraciones de ajuste fino que prueba la búsqueda |
| `EPOCAS_BUSQUEDA` | 8 | Épocas máximas de cada prueba de ajuste fino |
| `EPOCAS` | 20 | Épocas del ajuste fino final |
| `N_REMUESTREOS` | 2000 | Remuestreos *bootstrap* para los intervalos |

In [ ]:
import os
variable_entorno = os.environ.get          # las variables MA_* solo se usan en la prueba local reducida

SEMILLAS = tuple(int(s) for s in variable_entorno("MA_SEMILLAS", "42,7,2026").split(","))
EXTRACTORES = tuple(variable_entorno("MA_EXTRACTORES", "vit_small_patch16_dinov3.lvd1689m,vit_base_patch16_dinov3.lvd1689m,"
                                     "vit_large_patch16_dinov3.lvd1689m,vit_large_patch16_dinov3.sat493m").split(","))
PRUEBAS_CABEZA = int(variable_entorno("MA_PRUEBAS_CABEZA", "40"))
PRUEBAS_AJUSTE = int(variable_entorno("MA_PRUEBAS", "16"))
EPOCAS_BUSQUEDA = int(variable_entorno("MA_EPOCAS_BUSQUEDA", "8"))
EPOCAS = int(variable_entorno("MA_EPOCAS", "20"))
EPOCAS_CABEZA = int(variable_entorno("MA_EPOCAS_CABEZA", "300"))
N_REMUESTREOS = int(variable_entorno("MA_REMUESTREOS", "2000"))
LIMITE_GALAXIAS = int(variable_entorno("MA_LIMITE", "0"))
PREENTRENADO = variable_entorno("MA_PREENTRENADO", "1") == "1"
LADO_ENTRADA = int(variable_entorno("MA_LADO", "288"))
LADO_ALTERNATIVO = int(variable_entorno("MA_LADO_ALTERNATIVO", "224"))

from pathlib import Path
BASE_DRIVE = Path(variable_entorno("MA_BASE", "/content/drive/MyDrive"))
CARPETA_RESULTADOS = "resultados_modelos_alternativos"
NOMBRE_MODELO = "modelo4_dinov3_grz_redshift"
OBJETIVOS = ("log_mstar", "log_sfr", "d4000")
VARIABLES_ESCALARES = ("redshift",)
NOMBRES = {"log_mstar": "log M* [dex]", "log_sfr": "log SFR [dex]", "d4000": "D4000"}
CORTO = {"log_mstar": "M*", "log_sfr": "SFR", "d4000": "D4000"}
MARGEN = {"M*": 0.02, "SFR": 0.02, "D4000": 0.01}
CORTE_SSFR = -10.8
ENTRADA_BASE = {"representacion": "grz_estandarizado", "lado": LADO_ENTRADA, "modo_lado": "recorte"}
NOMBRE_CORTO_EXTRACTOR = {e: e.replace("vit_", "ViT-").replace("_patch16_dinov3", "").replace(".lvd1689m", "")
                             .replace(".sat493m", " satelital").replace("small", "S").replace("base", "B").replace("large", "L")
                          for e in EXTRACTORES}

## 3. Entorno

Descargamos el código del repositorio del equipo, de modo que cada resultado quede ligado a una versión concreta, e instalamos las bibliotecas que Colab no incluye: `timm` (modelos preentrenados) y `optuna` (búsqueda de hiperparámetros). Los datos y los resultados están en Google Drive. Este cuaderno necesita una GPU; recomendamos una A100.

In [ ]:
import sys, json, time, subprocess, platform, warnings
from pathlib import Path
from dataclasses import replace

EN_COLAB = "google.colab" in sys.modules
URL_REPOSITORIO = "https://github.com/erteck/InferenciaPropiedadesFisicas.git"
CARPETA_REPOSITORIO = Path(variable_entorno("MA_CARPETA_REPOSITORIO", "/content/InferenciaPropiedadesFisicas"))
if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm>=1.0.20", "optuna>=4", "hickle"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    if not CARPETA_REPOSITORIO.exists():
        subprocess.run(["git", "clone", "-q", URL_REPOSITORIO, str(CARPETA_REPOSITORIO)], check=True)
    subprocess.run(["git", "-C", str(CARPETA_REPOSITORIO), "pull", "-q"], check=True)
sys.path.insert(0, str(CARPETA_REPOSITORIO))
COMMIT = subprocess.run(["git", "-C", str(CARPETA_REPOSITORIO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()

warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch, timm, optuna
import matplotlib.pyplot as plt
from IPython.display import display

from src.modelos_alternativos import arquitecturas, busqueda, evaluacion, experimento, interpretacion
from src.modelos_alternativos.arquitecturas import EspecificacionModelo
from src.modelos_alternativos.contexto import preparar_contexto
from src.modelos_alternativos.entrenamiento import ConfiguracionEntrenamiento
from src.katachi_fits.diagnostico import radio_mitad_luz

DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
pd.set_option("display.precision", 4, "display.max_columns", 40, "display.width", 220)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
print("Python", platform.python_version(), "· commit", COMMIT[:12], "· timm", timm.__version__, "· optuna", optuna.__version__,
      "·", DISPOSITIVO, torch.cuda.get_device_name(0) if DISPOSITIVO.type == "cuda" else "")

## 4. Datos

Preparamos la muestra y las transformaciones con las reglas del protocolo común. Todos los parámetros (la escala de compresión de cada banda, las medias y desviaciones de la imagen, de las etiquetas y del redshift) se calculan solo con las galaxias de entrenamiento y se aplican sin cambios a validación y prueba. Los conteos de la muestra se calculan aquí, no se imponen.

In [ ]:
CONTEXTO = preparar_contexto(BASE_DRIVE, CARPETA_RESULTADOS, OBJETIVOS, VARIABLES_ESCALARES, limite_galaxias=LIMITE_GALAXIAS)
CARPETA_MODELO = CONTEXTO.guardia.mkdir(BASE_DRIVE / CARPETA_RESULTADOS / NOMBRE_MODELO)
MUESTRA, INDICES = CONTEXTO.muestra, CONTEXTO.indices
TIPO_GALAXIA = np.where(MUESTRA.log_ssfr < CORTE_SSFR, "apagada", "formadora")
display(pd.crosstab(MUESTRA.particion, pd.Series(TIPO_GALAXIA, name="tipo de galaxia", index=MUESTRA.index), margins=True,
                    margins_name="total").loc[["Train", "Validation", "Test", "total"]])
if len(CONTEXTO.excluidas):
    display(CONTEXTO.excluidas.groupby(["particion", "motivo"]).size().rename("galaxias").reset_index())
t = CONTEXTO.transformacion_grz
display(pd.DataFrame({"s_b (percentil 75 de |f|)": t.escala_asinh, "media tras asinh": t.media, "desviación tras asinh": t.desviacion},
                     index=["g", "r", "z"]).round(5))
CONTEXTO.guardia.write_json(CARPETA_MODELO / "contexto_datos.json", CONTEXTO.resumen(), allow_replace=True)
RADIO_MITAD_LUZ = radio_mitad_luz(CONTEXTO.imagenes, banda=1)          # en píxeles de la imagen de 300 px, banda r
print(f"Radio de mitad de luz (banda r): mediana {np.median(RADIO_MITAD_LUZ):.1f} px")

Las galaxias de la muestra se ven así a la entrada de la red. La imagen muestra los tres canales estandarizados (z, r y g como rojo, verde y azul) en el recorte central de 288 × 288 píxeles, que es la entrada de DINOv3. El círculo marca el radio que encierra la mitad de la luz.

In [ ]:
PREPARADOR_VISTA = experimento.crear_preparador(CONTEXTO, ENTRADA_BASE)
entrenamiento = MUESTRA.iloc[INDICES["Train"]]
ejemplos = [entrenamiento.index[np.argmin(np.abs(entrenamiento.log_mstar.to_numpy() - q))]
            for q in np.quantile(entrenamiento.log_mstar, np.linspace(0.05, 0.95, 6))]
fig, ejes = plt.subplots(1, 6, figsize=(15, 2.9))
for eje, i in zip(ejes, ejemplos):
    x = PREPARADOR_VISTA(torch.from_numpy(CONTEXTO.imagenes[i:i + 1].astype(np.float32)))[0].numpy()[[2, 1, 0]]
    if PREPARADOR_VISTA.representacion == "rgb_lupton":
        x = x[[2, 1, 0]]
    rgb = np.transpose(x, (1, 2, 0)); rgb = np.clip((rgb - np.percentile(rgb, 1)) / (np.percentile(rgb, 99.7) - np.percentile(rgb, 1)), 0, 1)
    eje.imshow(rgb, origin="lower")
    escala = x.shape[-1] / 300 if PREPARADOR_VISTA.modo_lado == "reescala" else 1
    eje.add_patch(plt.Circle(((x.shape[-1] - 1) / 2,) * 2, RADIO_MITAD_LUZ[i] * escala, fill=False, color="w", lw=0.8, ls="--"))
    g = MUESTRA.loc[i]
    eje.set_title(f"log M* {g.log_mstar:.1f} · log SFR {g.log_sfr:.1f}\nD4000 {g.d4000:.2f} · z {g.redshift:.3f}", fontsize=7)
    eje.set_xticks([]); eje.set_yticks([])
plt.suptitle(f"Entrada de la red ({x.shape[-1]} × {x.shape[-1]} px)", y=1.04); plt.show()

In [ ]:
# Utilidades para resumir corridas con las métricas del protocolo común.
def metricas_corrida(corrida, particion="Validation", tta=False):
    prediccion = experimento.predicciones_originales(CONTEXTO, corrida, particion, DISPOSITIVO, tta=tta)
    return experimento.tabla_corrida(CONTEXTO, prediccion, particion, corrida=corrida.identificador), prediccion

def resumen_corridas(corridas, etiqueta, particion="Validation"):
    # Media y desviación entre semillas de cada métrica; fila ancha por modelo.
    tablas = pd.concat([metricas_corrida(r, particion)[0].assign(semilla=r.resumen["configuracion"]["semilla"]) for r in corridas])
    fila = {"modelo": etiqueta, "semillas": len(corridas),
            "MSE_normalizado_medio": tablas.groupby("semilla").MSE_normalizado.mean().mean(),
            "desviacion_entre_semillas": tablas.groupby("semilla").MSE_normalizado.mean().std(ddof=1) if len(corridas) > 1 else np.nan}
    for objetivo, g in tablas.groupby("objetivo", sort=False):
        for m in ("RMSE", "MAE", "sesgo", "R2"):
            fila[f"{m} {CORTO[objetivo]}"] = g[m].mean()
    fila["minutos_entrenamiento"] = np.mean([r.resumen["minutos_entrenamiento"] for r in corridas])
    fila["parametros_entrenables_M"] = corridas[0].resumen["parametros_entrenables"] / 1e6
    return fila, tablas

## 5. Etapa 1: representaciones congeladas

### 5.1 Tamaño del modelo y forma de presentar la imagen

Calculamos una vez el vector de cada galaxia de entrenamiento y validación con cada tamaño de DINOv3 y con dos formas de presentar la imagen:

- **Bandas estandarizadas:** la entrada del protocolo común (g, r y z con asinh y estandarización).
- **Imagen en color de Lupton:** la composición en color de Lupton et al. (2004), con z, r y g como rojo, verde y azul, normalizada con las medias y desviaciones con que se entrenó cada modelo. Es más parecida a las fotos que vio DINOv3, aunque pierde parte del rango de brillo.

De cada imagen guardamos el token de clase (CLS) y el promedio de los tokens de los parches. Encima entrenamos las tres cabezas del protocolo común con tres formas de resumir la imagen: solo CLS, solo el promedio de los parches o ambos unidos. Cada cabeza se entrena en segundos, porque los vectores ya están calculados.

In [ ]:
from src.modelos_alternativos.representaciones import ConfiguracionCabeza

CONFIG_CABEZA_BASE = ConfiguracionCabeza(unidades_ocultas=256, dropout=0.1, tasa_aprendizaje=1e-3, decaimiento_pesos=1e-2,
                                         epocas=EPOCAS_CABEZA, tamano_lote=256, perdida="huber", semilla=SEMILLAS[0])
REPRESENTACIONES, FILAS_SONDEO = {}, []
for extractor in EXTRACTORES:
    for representacion in ("grz_estandarizado", "rgb_lupton"):
        entrada = {**ENTRADA_BASE, "representacion": representacion}
        espec = EspecificacionModelo(nombre_extractor=extractor, preentrenado=PREENTRENADO, lado_entrada=LADO_ENTRADA,
                                     estrategia_ajuste="congelado")
        inicio = time.time()
        REPRESENTACIONES[(extractor, representacion)] = experimento.representaciones_en_cache(
            CONTEXTO, CARPETA_MODELO, espec, entrada, ("Train", "Validation"), DISPOSITIVO)
        minutos_extraccion = (time.time() - inicio) / 60
        for agregacion in ("cls", "promedio", "cls_y_promedio"):
            corrida = experimento.ejecutar_cabeza_congelada(
                CONTEXTO, CARPETA_MODELO, "sondeo", replace(espec, agregacion_tokens=agregacion), CONFIG_CABEZA_BASE, entrada,
                REPRESENTACIONES[(extractor, representacion)], DISPOSITIVO, registro=lambda *_: None)
            tabla, _ = metricas_corrida(corrida)
            FILAS_SONDEO.append({"extractor": NOMBRE_CORTO_EXTRACTOR[extractor], "nombre_timm": extractor, "representacion": representacion,
                                 "agregacion": agregacion, "MSE_normalizado_medio": tabla.MSE_normalizado.mean(),
                                 **{f"RMSE {CORTO[r.objetivo]}": r.RMSE for r in tabla.itertuples()},
                                 "minutos_extraccion": minutos_extraccion, "segundos_cabeza": corrida.resumen["minutos_entrenamiento"] * 60})
        torch.cuda.empty_cache()
SONDEO = pd.DataFrame(FILAS_SONDEO).sort_values("MSE_normalizado_medio").reset_index(drop=True)
CONTEXTO.guardia.write(CARPETA_MODELO / "sondeo_congelado.csv", lambda f: SONDEO.to_csv(f, index=False), allow_replace=True)
display(SONDEO.round(4))

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(15, 3.8), gridspec_kw={"width_ratios": [1.4, 1]})
orden_extractores = [NOMBRE_CORTO_EXTRACTOR[e] for e in EXTRACTORES]
ancho = 0.13
for j, (representacion, agregacion) in enumerate([(r, a) for r in ("grz_estandarizado", "rgb_lupton") for a in ("cls", "promedio", "cls_y_promedio")]):
    d = SONDEO[(SONDEO.representacion == representacion) & (SONDEO.agregacion == agregacion)].set_index("extractor").reindex(orden_extractores)
    ejes[0].bar(np.arange(len(orden_extractores)) + (j - 2.5) * ancho, d.MSE_normalizado_medio, ancho,
                label=f"{'bandas' if representacion == 'grz_estandarizado' else 'Lupton'} · {agregacion}",
                hatch="" if representacion == "grz_estandarizado" else "//", alpha=0.9)
ejes[0].axhline(0.127879, color="k", ls=":", lw=1, label="modelo 1 (ResNet50 ajustada)")
ejes[0].set(xticks=range(len(orden_extractores)), xticklabels=orden_extractores, ylabel="MSE normalizado medio (validación)",
            title="Representaciones congeladas: menor es mejor"); ejes[0].legend(fontsize=6.5, ncol=2)
mejor_por_extractor = SONDEO.groupby("extractor").MSE_normalizado_medio.min().reindex(orden_extractores)
parametros = {"ViT-S": 21, "ViT-B": 86, "ViT-L": 300, "ViT-L satelital": 300}
ejes[1].plot([parametros.get(e, np.nan) for e in mejor_por_extractor.index], mejor_por_extractor.values, "o-", color="#264653")
for e, v in mejor_por_extractor.items():
    ejes[1].annotate(e, (parametros.get(e, np.nan), v), fontsize=7, xytext=(4, 4), textcoords="offset points")
ejes[1].set(xscale="log", xlabel="millones de parámetros", ylabel="mejor MSE normalizado medio", title="¿Ayuda un modelo más grande?")
plt.tight_layout(); plt.show()
MEJOR_SONDEO = SONDEO.iloc[0]
print("Mejor combinación congelada:", dict(MEJOR_SONDEO[["extractor", "representacion", "agregacion", "MSE_normalizado_medio"]]))

_Se completa con los resultados de la corrida._

### 5.2 Resolución

DINOv3 se preentrenó sobre todo con imágenes de 224 a 256 px. Comparamos el recorte central de 288 px con la imagen completa de 300 px reducida a 224 px, que conserva todo el campo con menos detalle, para la mejor combinación de la sección anterior.

In [ ]:
EXTRACTOR_ELEGIDO, REPRESENTACION_ELEGIDA, AGREGACION_ELEGIDA = (str(MEJOR_SONDEO.nombre_timm), str(MEJOR_SONDEO.representacion),
                                                                 str(MEJOR_SONDEO.agregacion))
filas = []
for lado, modo in ((LADO_ENTRADA, "recorte"), (LADO_ALTERNATIVO, "reescala")):
    entrada = {"representacion": REPRESENTACION_ELEGIDA, "lado": lado, "modo_lado": modo}
    espec = EspecificacionModelo(nombre_extractor=EXTRACTOR_ELEGIDO, preentrenado=PREENTRENADO, lado_entrada=lado,
                                 estrategia_ajuste="congelado", agregacion_tokens=AGREGACION_ELEGIDA)
    reps = experimento.representaciones_en_cache(CONTEXTO, CARPETA_MODELO, espec, entrada, ("Train", "Validation"), DISPOSITIVO)
    corrida = experimento.ejecutar_cabeza_congelada(CONTEXTO, CARPETA_MODELO, "resolucion", espec, CONFIG_CABEZA_BASE, entrada, reps,
                                                    DISPOSITIVO, registro=lambda *_: None)
    tabla, _ = metricas_corrida(corrida)
    filas.append({"entrada": f"{'recorte central' if modo == 'recorte' else 'imagen completa reducida'} de {lado} px",
                  "lado": lado, "modo_lado": modo, "MSE_normalizado_medio": tabla.MSE_normalizado.mean(),
                  **{f"RMSE {CORTO[r.objetivo]}": r.RMSE for r in tabla.itertuples()}})
RESOLUCION = pd.DataFrame(filas).sort_values("MSE_normalizado_medio")
display(RESOLUCION.round(4))
ENTRADA_ELEGIDA = {"representacion": str(REPRESENTACION_ELEGIDA), "lado": int(RESOLUCION.iloc[0].lado), "modo_lado": str(RESOLUCION.iloc[0].modo_lado)}
ESPEC_CONGELADA = EspecificacionModelo(nombre_extractor=EXTRACTOR_ELEGIDO, preentrenado=PREENTRENADO, lado_entrada=ENTRADA_ELEGIDA["lado"],
                                       estrategia_ajuste="congelado", agregacion_tokens=AGREGACION_ELEGIDA)
REPS_ELEGIDAS = experimento.representaciones_en_cache(CONTEXTO, CARPETA_MODELO, ESPEC_CONGELADA, ENTRADA_ELEGIDA, ("Train", "Validation"),
                                                      DISPOSITIVO)
print("Entrada elegida:", ENTRADA_ELEGIDA)

_Se completa con los resultados de la corrida._

### 5.3 Ajuste de la cabeza

Con el extractor congelado, la única parte que aprende es la cabeza. Buscamos su mejor configuración con Optuna (muestreo bayesiano TPE): el número de unidades ocultas, el *dropout*, la tasa de aprendizaje, el *weight decay* y la pérdida. Como cada prueba tarda segundos, podemos explorar 40 configuraciones. La mejor se entrena después con las tres semillas.

In [ ]:
def proponer_cabeza(prueba):
    return {"unidades_ocultas": prueba.suggest_categorical("unidades_ocultas", [128, 256, 512, 1024]),
            "dropout": prueba.suggest_float("dropout", 0.0, 0.5),
            "tasa_aprendizaje": prueba.suggest_float("tasa_aprendizaje", 1e-4, 5e-3, log=True),
            "decaimiento_pesos": prueba.suggest_float("decaimiento_pesos", 1e-4, 0.3, log=True),
            "perdida": prueba.suggest_categorical("perdida", ["huber", "mse"])}

def evaluar_cabeza(valores, al_terminar_epoca):
    corrida = experimento.ejecutar_cabeza_congelada(CONTEXTO, CARPETA_MODELO, "busqueda_cabeza", ESPEC_CONGELADA,
                                                    replace(CONFIG_CABEZA_BASE, **valores), ENTRADA_ELEGIDA, REPS_ELEGIDAS,
                                                    DISPOSITIVO, guardar=False, construir_modelo_imagen=False,
                                                    registro=lambda *_: None)
    return corrida.resumen["mejor_criterio_validacion"]

respaldo = lambda origen, destino: CONTEXTO.guardia.write(destino, lambda f: Path(f).write_bytes(Path(origen).read_bytes()), allow_replace=True)
sufijo = f"_lim{LIMITE_GALAXIAS}" if LIMITE_GALAXIAS else ""
ESTUDIO_CABEZA = busqueda.ejecutar_busqueda(f"{NOMBRE_MODELO}_cabeza{sufijo}", CARPETA_MODELO / "busqueda_cabeza.db", proponer_cabeza,
                                            evaluar_cabeza, PRUEBAS_CABEZA, semilla=SEMILLAS[0], guardar_respaldo=respaldo)
PRUEBAS_CABEZA_TABLA = busqueda.tabla_pruebas(ESTUDIO_CABEZA)
display(PRUEBAS_CABEZA_TABLA.head(8).round(5))
MEJOR_CABEZA = {k: PRUEBAS_CABEZA_TABLA.iloc[0][k] for k in ("unidades_ocultas", "dropout", "tasa_aprendizaje", "decaimiento_pesos", "perdida")}
MEJOR_CABEZA = {"unidades_ocultas": int(MEJOR_CABEZA["unidades_ocultas"]), "dropout": float(MEJOR_CABEZA["dropout"]),
                "tasa_aprendizaje": float(MEJOR_CABEZA["tasa_aprendizaje"]), "decaimiento_pesos": float(MEJOR_CABEZA["decaimiento_pesos"]),
                "perdida": str(MEJOR_CABEZA["perdida"])}
CORRIDAS_CONGELADAS = [experimento.ejecutar_cabeza_congelada(CONTEXTO, CARPETA_MODELO, "congelado_final", ESPEC_CONGELADA,
                                                             replace(CONFIG_CABEZA_BASE, **MEJOR_CABEZA, semilla=s), ENTRADA_ELEGIDA,
                                                             REPS_ELEGIDAS, DISPOSITIVO) for s in SEMILLAS]
FILA_CONGELADA, _ = resumen_corridas(CORRIDAS_CONGELADAS, f"4a · DINOv3 {NOMBRE_CORTO_EXTRACTOR[EXTRACTOR_ELEGIDO]} congelado")
importancia = busqueda.importancia_hiperparametros(ESTUDIO_CABEZA)
terminadas = PRUEBAS_CABEZA_TABLA.sort_values("prueba")
fig, ejes = plt.subplots(1, 2, figsize=(12, 3.4))
ejes[0].scatter(terminadas.prueba, terminadas.criterio_validacion, s=16); ejes[0].plot(terminadas.prueba, terminadas.criterio_validacion.cummin(), "C3")
ejes[0].axhline(SONDEO.iloc[0].MSE_normalizado_medio, color="k", ls=":", lw=1, label="cabeza por defecto"); ejes[0].legend(fontsize=7)
ejes[0].set(xlabel="prueba", ylabel="criterio de validación", title="Búsqueda de la cabeza")
if len(importancia):
    ejes[1].barh(importancia.index, importancia.values, color="#264653"); ejes[1].invert_yaxis()
ejes[1].set(xlabel="importancia (fANOVA)", title="Qué hiperparámetros de la cabeza pesan más")
plt.tight_layout(); plt.show()
print(f"DINOv3 congelado con la mejor cabeza: {FILA_CONGELADA['MSE_normalizado_medio']:.4f} ± {FILA_CONGELADA['desviacion_entre_semillas']:.4f}")

_Se completa con los resultados de la corrida._

## 6. Etapa 2: ajuste fino

Ahora permitimos que el extractor también aprenda. Con 7,000 galaxias de entrenamiento, ajustar todos los parámetros de un ViT puede sobreajustar, así que comparamos dos estrategias que adaptan solo una parte:

- **Abrir los últimos bloques.** Los primeros bloques del ViT, que detectan patrones generales, quedan fijos, y solo aprenden los últimos *N* bloques, con una tasa que decrece hacia los bloques más profundos (decaimiento por capa).
- **LoRA** (Hu et al. 2022). Todos los pesos originales quedan fijos y a cada capa de atención se le suma una corrección de bajo rango, B · A, que es la única parte que aprende. Con rango 8 en un ViT-B, se entrenan menos del 1 % de los parámetros.

La búsqueda de Optuna elige la estrategia y sus hiperparámetros a la vez. Cada prueba entrena hasta 8 épocas con la semilla 42, y las poco prometedoras se detienen antes (*successive halving*). La mejor configuración se entrena después con 20 épocas y las tres semillas.

| Hiperparámetro | Espacio de búsqueda |
|---|---|
| Estrategia | últimos bloques o LoRA |
| Bloques abiertos | 2, 4, 6 u 8 (solo para la primera estrategia) |
| Rango de LoRA | 4, 8, 16 o 32 (solo para LoRA) |
| Tasa del extractor | 10⁻⁵ a 10⁻³ (logarítmica) |
| Tasa de la cabeza | 1 a 20 veces la del extractor |
| *Weight decay* | 10⁻³ a 0.1 (logarítmica) |
| Decaimiento por capa | 0.65 a 1.0 |
| *Stochastic depth* | 0 a 0.2 |

In [ ]:
ES_GRANDE = "large" in EXTRACTOR_ELEGIDO
CONFIG_AJUSTE_BASE = ConfiguracionEntrenamiento(epocas=EPOCAS, epocas_calentamiento=2, tamano_lote=32, perdida="huber",
                                                canales_ultimos=False, checkpoint_gradientes=ES_GRANDE, semilla=SEMILLAS[0])

def proponer_ajuste(prueba):
    estrategia = prueba.suggest_categorical("estrategia_ajuste", ["ultimos_bloques", "lora"])
    valores = {"estrategia_ajuste": estrategia,
               "tasa_extractor": prueba.suggest_float("tasa_extractor", 1e-5, 1e-3, log=True),
               "factor_tasa_cabeza": prueba.suggest_float("factor_tasa_cabeza", 1.0, 20.0, log=True),
               "decaimiento_pesos": prueba.suggest_float("decaimiento_pesos", 1e-3, 0.1, log=True),
               "decaimiento_por_capa": prueba.suggest_float("decaimiento_por_capa", 0.65, 1.0),
               "tasa_drop_path": prueba.suggest_float("tasa_drop_path", 0.0, 0.2)}
    if estrategia == "ultimos_bloques":
        valores["bloques_abiertos"] = prueba.suggest_categorical("bloques_abiertos", [2, 4, 6, 8])
    else:
        valores["rango_lora"] = prueba.suggest_categorical("rango_lora", [4, 8, 16, 32])
    return valores

def configuracion_ajuste(valores, epocas, semilla):
    espec = EspecificacionModelo(nombre_extractor=EXTRACTOR_ELEGIDO, preentrenado=PREENTRENADO, lado_entrada=ENTRADA_ELEGIDA["lado"],
                                 agregacion_tokens=AGREGACION_ELEGIDA, estrategia_ajuste=str(valores["estrategia_ajuste"]),
                                 bloques_abiertos=int(valores.get("bloques_abiertos") or 0), rango_lora=int(valores.get("rango_lora") or 8),
                                 alfa_lora=2.0 * float(valores.get("rango_lora") or 8), tasa_drop_path=float(valores["tasa_drop_path"]),
                                 unidades_ocultas=MEJOR_CABEZA["unidades_ocultas"], dropout_cabeza=MEJOR_CABEZA["dropout"])
    config = replace(CONFIG_AJUSTE_BASE, epocas=epocas, semilla=semilla, tasa_extractor=float(valores["tasa_extractor"]),
                     tasa_cabeza=float(valores["tasa_extractor"] * valores["factor_tasa_cabeza"]),
                     decaimiento_pesos=float(valores["decaimiento_pesos"]), decaimiento_por_capa=float(valores["decaimiento_por_capa"]),
                     epocas_calentamiento=min(2, max(1, epocas - 1)))
    return espec, config

def evaluar_ajuste(valores, al_terminar_epoca):
    espec, config = configuracion_ajuste(valores, EPOCAS_BUSQUEDA, SEMILLAS[0])
    corrida = experimento.ejecutar_corrida(CONTEXTO, CARPETA_MODELO, "busqueda_ajuste", espec, config, ENTRADA_ELEGIDA, DISPOSITIVO,
                                           al_terminar_epoca=al_terminar_epoca, guardar=False, registro=lambda *_: None)
    valor = corrida.resumen["mejor_criterio_validacion"]
    del corrida; torch.cuda.empty_cache()
    return valor

inicio = time.time()
ESTUDIO_AJUSTE = busqueda.ejecutar_busqueda(f"{NOMBRE_MODELO}_ajuste{sufijo}", CARPETA_MODELO / "busqueda_ajuste.db", proponer_ajuste,
                                            evaluar_ajuste, PRUEBAS_AJUSTE, semilla=SEMILLAS[0], epocas_minimas_antes_de_podar=2,
                                            guardar_respaldo=respaldo)
PRUEBAS_AJUSTE_TABLA = busqueda.tabla_pruebas(ESTUDIO_AJUSTE)
CONTEXTO.guardia.write(CARPETA_MODELO / "pruebas_ajuste.csv", lambda f: PRUEBAS_AJUSTE_TABLA.to_csv(f, index=False), allow_replace=True)
print(f"Búsqueda de ajuste fino: {len(PRUEBAS_AJUSTE_TABLA)} pruebas en {(time.time() - inicio) / 60:.0f} min")
display(PRUEBAS_AJUSTE_TABLA.head(10).round(5))

In [ ]:
terminadas = PRUEBAS_AJUSTE_TABLA[PRUEBAS_AJUSTE_TABLA.estado == "complete"].sort_values("prueba")
fig, ejes = plt.subplots(1, 3, figsize=(16, 3.8))
for estrategia, g in terminadas.groupby("estrategia_ajuste"):
    ejes[0].scatter(g.prueba, g.criterio_validacion, s=20, label=estrategia)
ejes[0].plot(terminadas.prueba, terminadas.criterio_validacion.cummin(), color="C3", lw=1, label="mejor hasta el momento")
ejes[0].axhline(FILA_CONGELADA["MSE_normalizado_medio"], color="k", ls=":", lw=1, label="DINOv3 congelado")
ejes[0].set(xlabel="prueba", ylabel="criterio de validación", title="Historia de la búsqueda"); ejes[0].legend(fontsize=7)
for t in ESTUDIO_AJUSTE.trials:
    if t.intermediate_values:
        e, v = zip(*sorted(t.intermediate_values.items()))
        ejes[1].plot(e, v, color="C3" if t.state.name == "PRUNED" else "C0", alpha=0.5, lw=0.8)
ejes[1].set(xlabel="época", ylabel="criterio de validación", yscale="log", title="Curvas por prueba (rojo: podadas)")
importancia = busqueda.importancia_hiperparametros(ESTUDIO_AJUSTE)
if len(importancia):
    ejes[2].barh(importancia.index, importancia.values, color="#264653"); ejes[2].invert_yaxis()
ejes[2].set(xlabel="importancia (fANOVA)", title="Qué hiperparámetros pesan más")
plt.tight_layout(); plt.show()

_Se completa con los resultados de la corrida._

In [ ]:
claves_ajuste = ["estrategia_ajuste", "tasa_extractor", "factor_tasa_cabeza", "decaimiento_pesos", "decaimiento_por_capa", "tasa_drop_path",
                 "bloques_abiertos", "rango_lora"]
mejor_ajuste = PRUEBAS_AJUSTE_TABLA[PRUEBAS_AJUSTE_TABLA.estado == "complete"].iloc[0]
VALORES_AJUSTE = {k: (None if k not in mejor_ajuste or pd.isna(mejor_ajuste[k]) else mejor_ajuste[k]) for k in claves_ajuste}
CORRIDAS_AJUSTADAS = []
for s in SEMILLAS:
    espec, config = configuracion_ajuste(VALORES_AJUSTE, EPOCAS, s)
    CORRIDAS_AJUSTADAS.append(experimento.ejecutar_corrida(CONTEXTO, CARPETA_MODELO, "ajuste_final", espec, config, ENTRADA_ELEGIDA, DISPOSITIVO))
FILA_AJUSTADA, TABLAS_AJUSTADAS = resumen_corridas(CORRIDAS_AJUSTADAS, f"4b · DINOv3 {NOMBRE_CORTO_EXTRACTOR[EXTRACTOR_ELEGIDO]} con ajuste fino "
                                                   f"({VALORES_AJUSTE['estrategia_ajuste']})")
print(f"Ajuste fino: {FILA_AJUSTADA['MSE_normalizado_medio']:.4f} ± {FILA_AJUSTADA['desviacion_entre_semillas']:.4f} · "
      f"congelado: {FILA_CONGELADA['MSE_normalizado_medio']:.4f} · parámetros entrenables: {FILA_AJUSTADA['parametros_entrenables_M']:.2f} M")

_Se completa con los resultados de la corrida._

## 7. Comparación en validación

La tabla ordena los modelos por la métrica principal del protocolo común, el promedio del MSE normalizado, y muestra además el RMSE, el MAE, el sesgo y el R² de cada propiedad, el tiempo de entrenamiento y el número de parámetros entrenables. Como referencia se incluyen los dos modelos que el equipo ya evaluó con el mismo protocolo, con las cifras que reporta su cuaderno, y el baseline del Avance 3, recalculado sobre nuestras galaxias de validación cuando sus predicciones están en Drive.

In [ ]:
REFERENCIAS = pd.DataFrame([
    {"modelo": "1 · ResNet50 multitarea, GRZ + redshift + Huber (equipo, semilla 42)", "semillas": 1, "MSE_normalizado_medio": 0.127879,
     "RMSE M*": 0.108829, "MAE M*": 0.082898, "R2 M*": 0.972454, "RMSE SFR": 0.597520, "MAE SFR": 0.394586, "R2 SFR": 0.736249,
     "RMSE D4000": 0.073037, "MAE D4000": 0.052342, "R2 D4000": 0.903949, "fuente": "reportado (1,261 galaxias)"},
    {"modelo": "2 · SVR sobre características de ImageNet + PCA + redshift (equipo)", "semillas": 1, "MSE_normalizado_medio": 0.442091,
     "RMSE M*": 0.266115, "MAE M*": 0.199477, "R2 M*": 0.835294, "RMSE SFR": 0.982788, "MAE SFR": 0.759899, "R2 SFR": 0.286476,
     "RMSE D4000": 0.161113, "MAE D4000": 0.125858, "R2 D4000": 0.532613, "fuente": "reportado (1,261 galaxias)"}])

def fila_baseline_avance3():
    # Cadena de ResNet50 del Avance 3 (solo imagen), recalculada sobre nuestras galaxias de validación.
    ruta = BASE_DRIVE / "resultados_katachi_fits" / "avance3" / "predicciones_B2_seed42.csv"
    if not ruta.exists():
        return None, None
    pred = pd.read_csv(ruta, dtype={"mangaid": str}).set_index("mangaid")
    ids = MUESTRA.mangaid.iloc[INDICES["Validation"]]
    p = pred.loc[ids, [f"pred_{t}" for t in OBJETIVOS]].to_numpy(float)
    tabla = experimento.tabla_corrida(CONTEXTO, p, "Validation")
    fila = {"modelo": "Baseline del Avance 3 · cadena de ResNet50, solo imagen (semilla 42)", "semillas": 1,
            "MSE_normalizado_medio": tabla.MSE_normalizado.mean(), "fuente": "recalculado"}
    for _, r in tabla.iterrows():
        for m in ("RMSE", "MAE", "sesgo", "R2"):
            fila[f"{m} {CORTO[r.objetivo]}"] = r[m]
    return fila, p

FILA_AVANCE3, PRED_AVANCE3_VALIDACION = fila_baseline_avance3()
filas = [FILA_CONGELADA, FILA_AJUSTADA]
COMPARATIVA = pd.concat([pd.DataFrame(filas).assign(fuente="este cuaderno"), REFERENCIAS,
                         pd.DataFrame([FILA_AVANCE3]) if FILA_AVANCE3 else None]).sort_values("MSE_normalizado_medio").reset_index(drop=True)
COMPARATIVA.index = COMPARATIVA.index + 1
columnas = ["modelo", "MSE_normalizado_medio", "desviacion_entre_semillas", "RMSE M*", "RMSE SFR", "RMSE D4000", "MAE M*", "MAE SFR",
            "MAE D4000", "sesgo M*", "sesgo SFR", "sesgo D4000", "R2 M*", "R2 SFR", "R2 D4000", "minutos_entrenamiento",
            "parametros_entrenables_M", "semillas", "fuente"]
display(COMPARATIVA[[c for c in columnas if c in COMPARATIVA]].round(4))
CONTEXTO.guardia.write(CARPETA_MODELO / "comparativa_validacion.csv", lambda f: COMPARATIVA.to_csv(f, index=False), allow_replace=True)

fig, ejes = plt.subplots(1, 4, figsize=(17, 0.6 + 0.45 * len(COMPARATIVA)), sharey=True)
etiquetas = [m if len(m) < 60 else m[:57] + "…" for m in COMPARATIVA.modelo]
for eje, columna in zip(ejes, ["MSE_normalizado_medio", "RMSE M*", "RMSE SFR", "RMSE D4000"]):
    colores = ["#264653" if f == "este cuaderno" else "0.65" for f in COMPARATIVA.fuente]
    eje.barh(etiquetas, COMPARATIVA[columna], color=colores); eje.set_xlabel(columna)
ejes[0].invert_yaxis(); plt.suptitle("Comparación en validación (oscuro: modelos de este cuaderno)", y=1.02); plt.tight_layout(); plt.show()

_Se completa con los resultados de la corrida._

In [ ]:
# Elección dentro de este cuaderno: la versión con menor criterio medio entre semillas.
if FILA_AJUSTADA["MSE_normalizado_medio"] <= FILA_CONGELADA["MSE_normalizado_medio"]:
    CORRIDAS_ELEGIDAS, NOMBRE_ELEGIDA = CORRIDAS_AJUSTADAS, f"{NOMBRE_CORTO_EXTRACTOR[EXTRACTOR_ELEGIDO]} con ajuste fino"
else:
    CORRIDAS_ELEGIDAS, NOMBRE_ELEGIDA = CORRIDAS_CONGELADAS, f"{NOMBRE_CORTO_EXTRACTOR[EXTRACTOR_ELEGIDO]} congelado"
print("Versión que representa al modelo 4:", NOMBRE_ELEGIDA)

## 8. Diagnóstico del modelo elegido

Revisamos el ajuste del modelo elegido con tres herramientas: la evolución del error por época en entrenamiento y validación, la comparación de las métricas entre las dos particiones y los residuos (predicción − etiqueta) frente al valor real y al redshift. Una brecha grande entre entrenamiento y validación indica sobreajuste; errores altos en ambas, junto con una pendiente negativa del residuo frente al valor real (regresión a la media), indican subajuste.

In [ ]:
CORRIDA_ELEGIDA = CORRIDAS_ELEGIDAS[0]
historial = CORRIDA_ELEGIDA.historial
if "perdida_entrenamiento" not in historial:
    print("El modelo elegido usa el extractor congelado: solo la cabeza aprendió y no hay curvas de entrenamiento de la imagen.")
else:
  fig, ejes = plt.subplots(1, 4, figsize=(17, 3.4))
  ejes[0].plot(historial.epoca, historial.perdida_entrenamiento, label="pérdida de entrenamiento")
  ejes[0].plot(historial.epoca, historial.criterio_validacion, label="criterio de validación")
  ejes[0].axvline(CORRIDA_ELEGIDA.resumen["mejor_epoca"], color="k", ls=":", lw=1, label="época elegida")
  ejes[0].set(xlabel="época", yscale="log", title="Curvas de entrenamiento"); ejes[0].legend(fontsize=7)
  for k, (eje, t) in enumerate(zip(ejes[1:], OBJETIVOS)):
      eje.plot(historial.epoca, np.sqrt(historial[f"mse_validacion_salida{k}"]) * CONTEXTO.desviacion_etiquetas_entrenamiento[k], color="#264653")
      eje.set(xlabel="época", ylabel="RMSE en validación", title=NOMBRES[t])
  plt.tight_layout(); plt.show()

ajuste = pd.concat([metricas_corrida(CORRIDA_ELEGIDA, p)[0] for p in ("Train", "Validation")])
ajuste = ajuste.pivot_table(index="objetivo", columns="particion", values=["RMSE", "sesgo", "sigma_NMAD"]).reindex(list(OBJETIVOS))
ajuste[("cociente", "Validation / Train")] = ajuste[("RMSE", "Validation")] / ajuste[("RMSE", "Train")]
ajuste[("referencia", "error de la etiqueta")] = [MUESTRA[f"error_{t}"].median() for t in OBJETIVOS]
display(ajuste.round(4))

_Se completa con los resultados de la corrida._

In [ ]:
PRED_VALIDACION = experimento.predicciones_originales(CONTEXTO, CORRIDA_ELEGIDA, "Validation", DISPOSITIVO)
Y_VALIDACION = CONTEXTO.etiquetas("Validation")
z_validacion = MUESTRA.redshift.to_numpy()[INDICES["Validation"]]

def mediana_movil(x, r, n=10):
    bordes = np.quantile(x, np.linspace(0, 1, n + 1)); centro, med, p16, p84 = [], [], [], []
    for a, b in zip(bordes[:-1], bordes[1:]):
        s = (x >= a) & (x <= b)
        if s.sum() >= 10:
            centro.append(np.median(x[s])); med.append(np.median(r[s])); p16.append(np.percentile(r[s], 16)); p84.append(np.percentile(r[s], 84))
    return map(np.array, (centro, med, p16, p84))

fig, ejes = plt.subplots(3, 3, figsize=(14, 11))
for k, t in enumerate(OBJETIVOS):
    y, p = Y_VALIDACION[:, k], PRED_VALIDACION[:, k]
    lim = np.percentile(np.r_[y, p], [0.5, 99.5])
    ejes[0, k].hexbin(y, p, gridsize=45, cmap="viridis", mincnt=1, bins="log", extent=(*lim, *lim))
    ejes[0, k].plot(lim, lim, "k-", lw=0.8)
    m = evaluacion.metricas(y, p)
    ejes[0, k].text(0.03, 0.97, f"RMSE {m['RMSE']:.3f}\nsesgo {m['sesgo']:+.3f}\nσ_NMAD {m['sigma_NMAD']:.3f}", transform=ejes[0, k].transAxes,
                    va="top", fontsize=8, bbox=dict(boxstyle="round", facecolor="white", alpha=0.85, edgecolor="0.7"))
    ejes[0, k].set(xlim=lim, ylim=lim, xlabel=f"{NOMBRES[t]} (etiqueta)", ylabel="predicción")
    for fila, x, nombre_x in ((1, y, f"{NOMBRES[t]} (etiqueta)"), (2, z_validacion, "redshift")):
        cx, med, p16, p84 = mediana_movil(x, p - y)
        ejes[fila, k].plot(cx, med, "o-", ms=3, color="#264653"); ejes[fila, k].fill_between(cx, p16, p84, color="#264653", alpha=0.2)
        ejes[fila, k].axhline(0, color="k", lw=0.6); ejes[fila, k].set(xlabel=nombre_x, ylabel="residuo")
plt.suptitle("Validación: predicción frente a etiqueta y mediana del residuo (franja: 68 % central)", y=1.0); plt.tight_layout(); plt.show()

tipo_validacion = TIPO_GALAXIA[INDICES["Validation"]]
display(evaluacion.metricas_por_grupo(Y_VALIDACION, PRED_VALIDACION, tipo_validacion, OBJETIVOS)
        .pivot(index="objetivo", columns="grupo", values=["RMSE", "sesgo"]).reindex(list(OBJETIVOS)).round(4))

_Se completa con los resultados de la corrida._

## 9. Qué ve DINOv3

Una propiedad conocida de DINO es que los vectores de sus parches agrupan regiones con contenido parecido aunque nunca se le hayan dado etiquetas. Para verlo, proyectamos los vectores de los parches de unas cuantas galaxias de validación en sus tres componentes principales y las mostramos como colores: parches del mismo color tienen representaciones parecidas. Si el modelo distingue el núcleo, el disco y el cielo, cada región debería tener un color propio.

In [ ]:
from sklearn.decomposition import PCA
red = CORRIDA_ELEGIDA.modelo.extractor.red
validacion = MUESTRA.iloc[INDICES["Validation"]]
elegidas = [validacion.index[np.argmin(np.abs(validacion.log_mstar.to_numpy() - q))] for q in np.quantile(validacion.log_mstar, [0.1, 0.4, 0.7, 0.95])]
x = CORRIDA_ELEGIDA.preparador(torch.from_numpy(CONTEXTO.imagenes[elegidas].astype(np.float32)).to(DISPOSITIVO))
with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16, enabled=DISPOSITIVO.type == "cuda"):
    tokens = red.forward_features(x).float()
lado_parches = x.shape[-1] // 16
parches = tokens[:, int(getattr(red, "num_prefix_tokens", 1)):].cpu().numpy()
componentes = PCA(3, random_state=0).fit_transform(parches.reshape(-1, parches.shape[-1])).reshape(len(elegidas), lado_parches, lado_parches, 3)
fig, ejes = plt.subplots(2, len(elegidas), figsize=(3.2 * len(elegidas), 6.4))
for j, i in enumerate(elegidas):
    imagen = x[j].float().cpu().numpy()
    imagen = imagen[[2, 1, 0]] if CORRIDA_ELEGIDA.preparador.representacion == "grz_estandarizado" else imagen
    rgb = np.transpose(imagen, (1, 2, 0)); rgb = np.clip((rgb - np.percentile(rgb, 1)) / (np.percentile(rgb, 99.7) - np.percentile(rgb, 1)), 0, 1)
    ejes[0, j].imshow(rgb, origin="lower"); ejes[0, j].set_title(f"log M* {MUESTRA.loc[i].log_mstar:.1f} · D4000 {MUESTRA.loc[i].d4000:.2f}", fontsize=8)
    c3 = componentes[j]; c3 = (c3 - componentes.min((0, 1, 2))) / (componentes.max((0, 1, 2)) - componentes.min((0, 1, 2)))
    ejes[1, j].imshow(c3, origin="lower", interpolation="nearest")
    for e in ejes[:, j]:
        e.set_xticks([]); e.set_yticks([])
ejes[1, 0].set_ylabel("3 componentes principales\nde los parches")
plt.tight_layout(); plt.show()

_Se completa con los resultados de la corrida._

## 10. ¿Qué información usa el modelo?

Medimos la importancia de cada entrada como el aumento del error cuando el modelo deja de recibirla, en las galaxias de validación. Quitamos una banda a la vez (la reemplazamos por el nivel del cielo), una región de la galaxia a la vez (anillos en unidades del radio de mitad de luz) y revolvemos el redshift entre galaxias. Las imágenes modificadas son distintas de las que el modelo vio al entrenar, así que la prueba indica qué información usa, no qué información existe. Con la misma prueba en todos los modelos podemos comparar cómo razonan.

In [ ]:
conjunto_validacion = CONTEXTO.conjunto("Validation", DISPOSITIVO)
radio_validacion = RADIO_MITAD_LUZ[INDICES["Validation"]]
base = CORRIDA_ELEGIDA.preparador
prediccion_base = CONTEXTO.a_escala_original(interpretacion.predecir_con_modificacion(CORRIDA_ELEGIDA.modelo, base, conjunto_validacion))
pruebas = [("banda", nombre, dict(canal=k)) for k, nombre in enumerate(["canal 1 (g)", "canal 2 (r)", "canal 3 (z)"]
                                                                     if base.representacion == "grz_estandarizado"
                                                                     else ["canal R (z)", "canal G (r)", "canal B (g)"])]
pruebas += [("región", f"{a}–{b} R50" if np.isfinite(b) else f"> {a} R50", dict(anillo=(a, b), radio_mitad_luz_px=radio_validacion))
            for a, b in [(0, 0.5), (0.5, 1), (1, 2), (2, 4), (4, np.inf)]]
pruebas += [("variable escalar", f"{nombre} revuelto", dict(escalar_permutado=k, semilla=SEMILLAS[0]))
            for k, nombre in enumerate(VARIABLES_ESCALARES)]
filas = []
for tipo, nombre, argumentos in pruebas:
    p = CONTEXTO.a_escala_original(interpretacion.predecir_con_modificacion(CORRIDA_ELEGIDA.modelo, base, conjunto_validacion, **argumentos))
    for k, t in enumerate(OBJETIVOS):
        d, lo, hi = evaluacion.diferencia_pareada_rmse(Y_VALIDACION[:, k], p[:, k], prediccion_base[:, k], n_remuestreos=N_REMUESTREOS)
        filas.append({"tipo": tipo, "entrada quitada": nombre, "propiedad": CORTO[t], "aumento del RMSE": d, "lo": lo, "hi": hi})
IMPORTANCIA = pd.DataFrame(filas)
CONTEXTO.guardia.write(CARPETA_MODELO / "importancia_entradas.csv", lambda f: IMPORTANCIA.to_csv(f, index=False), allow_replace=True)
fig, ejes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
for eje, t in zip(ejes, OBJETIVOS):
    d = IMPORTANCIA[IMPORTANCIA.propiedad == CORTO[t]]
    colores = d.tipo.map({"banda": "#2a9d8f", "región": "#264653", "variable escalar": "#e76f51"})
    eje.barh(d["entrada quitada"], d["aumento del RMSE"], xerr=[d["aumento del RMSE"] - d.lo, d.hi - d["aumento del RMSE"]],
             color=colores, capsize=2)
    eje.axvline(0, color="k", lw=0.6); eje.set(xlabel=f"aumento del RMSE · {NOMBRES[t]}")
ejes[0].invert_yaxis(); plt.suptitle("Importancia de cada entrada (verde: bandas; azul: regiones; naranja: redshift)", y=1.02)
plt.tight_layout(); plt.show()

_Se completa con los resultados de la corrida._

## 11. Evaluación en el conjunto de prueba

Con el modelo, su configuración y sus semillas ya fijados en validación, registramos la elección y abrimos el conjunto de prueba **una sola vez**. El registro guarda la configuración y una huella de las predicciones de validación; si el cuaderno se vuelve a ejecutar con otro modelo, avisa que esa evaluación ya no es independiente.

Reportamos las métricas del protocolo común con la semilla 42 y la media de las tres semillas, con y sin promedio de las 8 simetrías de cada imagen (*test-time augmentation*, TTA). Además comparamos, galaxia por galaxia, con el baseline del Avance 3 y con las predicciones publicadas de Katachi (Alfonzo et al. 2024) en las mismas galaxias.

In [ ]:
CORRIDAS_FINALES = CORRIDAS_ELEGIDAS
ELECCION = {"modelo": NOMBRE_MODELO, "descripcion": f'DINOv3 {NOMBRE_ELEGIDA}', "corridas": [r.identificador for r in CORRIDAS_FINALES],
            "commit": COMMIT}
experimento.registrar_eleccion_antes_de_prueba(CONTEXTO, CARPETA_MODELO / "registro_prueba.json", ELECCION,
                                               experimento.predicciones_originales(CONTEXTO, CORRIDAS_FINALES[0], "Validation", DISPOSITIVO))
Y_PRUEBA = CONTEXTO.etiquetas("Test")
tablas, PRED_PRUEBA = [], {}
for corrida in CORRIDAS_FINALES:
    semilla = corrida.resumen["configuracion"]["semilla"]
    for tta in (False, True):
        p = experimento.predicciones_originales(CONTEXTO, corrida, "Test", DISPOSITIVO, tta=tta)
        PRED_PRUEBA[(semilla, tta)] = p
        tablas.append(experimento.tabla_corrida(CONTEXTO, p, "Test", semilla=semilla, tta="sí" if tta else "no"))
    experimento.guardar_predicciones(CONTEXTO, CARPETA_MODELO, "Test", PRED_PRUEBA[(semilla, False)], f"predicciones_semilla{semilla}")
METRICAS_PRUEBA = pd.concat(tablas)
CONTEXTO.guardia.write(CARPETA_MODELO / "metricas_prueba.csv", lambda f: METRICAS_PRUEBA.to_csv(f, index=False), allow_replace=True)
resumen_prueba = (METRICAS_PRUEBA.groupby(["tta", "objetivo"], sort=False)[["RMSE", "MAE", "sesgo", "R2", "MSE_normalizado"]]
                  .agg(["mean", "std"]).round(4))
display(resumen_prueba)
for tta in ("no", "sí"):
    medio = METRICAS_PRUEBA[METRICAS_PRUEBA.tta == tta].groupby("semilla").MSE_normalizado.mean()
    print(f"MSE normalizado medio en prueba, TTA {tta}: {medio.mean():.4f} ± {medio.std(ddof=1) if len(medio) > 1 else float('nan'):.4f}")

_Se completa con los resultados de la corrida._

In [ ]:
ids_prueba = MUESTRA.mangaid.iloc[INDICES["Test"]].to_numpy()
referencias_prueba = {}
ruta_avance3 = BASE_DRIVE / "resultados_katachi_fits" / "avance3" / "predicciones_prueba_B2_seed42.csv"
if ruta_avance3.exists():
    tabla = pd.read_csv(ruta_avance3, dtype={"mangaid": str}).set_index("mangaid")
    referencias_prueba["Baseline del Avance 3 (solo imagen)"] = tabla.reindex(ids_prueba)[[f"pred_{t}" for t in OBJETIVOS]].to_numpy(float)
columnas_katachi = ["pred_mstar", "pred_sfr", "pred_d4000"]
if set(columnas_katachi) <= set(MUESTRA.columns):
    referencias_prueba["Katachi publicado (imágenes SDSS)"] = MUESTRA.iloc[INDICES["Test"]][columnas_katachi].to_numpy(float)
filas = []
for nombre, referencia in referencias_prueba.items():
    validas = np.isfinite(referencia).all(1)
    for semilla in SEMILLAS:
        propia = PRED_PRUEBA.get((semilla, False))
        if propia is None:
            continue
        for k, t in enumerate(OBJETIVOS):
            d, lo, hi = evaluacion.diferencia_pareada_rmse(Y_PRUEBA[validas, k], propia[validas, k], referencia[validas, k],
                                                           n_remuestreos=N_REMUESTREOS)
            filas.append({"comparado con": nombre, "semilla": semilla, "propiedad": CORTO[t], "galaxias": int(validas.sum()),
                          "ΔRMSE (este modelo − referencia)": d, "IC 95 %": f"[{lo:+.3f}, {hi:+.3f}]",
                          "veredicto": evaluacion.veredicto(lo, hi, MARGEN[CORTO[t]])})
COMPARACION_PRUEBA = pd.DataFrame(filas)
if len(COMPARACION_PRUEBA):
    display(COMPARACION_PRUEBA.set_index(["comparado con", "propiedad", "semilla"]).round(4))

_Se completa con los resultados de la corrida._

## 12. Resumen para la tabla comparativa del avance

Guardamos en Drive la fila de este modelo con la métrica principal y las métricas complementarias en validación y en prueba, el tiempo de entrenamiento, el tiempo de inferencia y el tamaño del modelo. El cuaderno que reúne los seis modelos lee estos archivos.

In [ ]:
fila, _ = resumen_corridas(CORRIDAS_FINALES, f'4 · DINOv3 {NOMBRE_ELEGIDA}')
prueba_semilla42 = METRICAS_PRUEBA[(METRICAS_PRUEBA.semilla == SEMILLAS[0]) & (METRICAS_PRUEBA.tta == "no")]
fila.update({"MSE_normalizado_prueba_semilla42": prueba_semilla42.MSE_normalizado.mean(),
             **{f"RMSE prueba {CORTO[r.objetivo]}": r.RMSE for r in prueba_semilla42.itertuples()},
             "ms_por_galaxia_inferencia": experimento.tiempo_inferencia_ms(CONTEXTO, CORRIDAS_FINALES[0], DISPOSITIVO),
             "parametros_totales_M": CORRIDAS_FINALES[0].resumen["parametros_totales"] / 1e6,
             "configuracion": json.dumps({"especificacion": CORRIDAS_FINALES[0].resumen["especificacion"],
                                          "configuracion": CORRIDAS_FINALES[0].resumen["configuracion"],
                                          "entrada": CORRIDAS_FINALES[0].resumen["entrada"]}, ensure_ascii=False)})
FILA_COMPARATIVA = pd.DataFrame([fila])
CONTEXTO.guardia.write(CARPETA_MODELO / "fila_comparativa.csv", lambda f: FILA_COMPARATIVA.to_csv(f, index=False), allow_replace=True)
display(FILA_COMPARATIVA.drop(columns=["configuracion"]).T.rename(columns={0: "valor"}))

## 13. Conclusiones

_Se completa con los resultados de la corrida._

## 14. Referencias

- Akiba, T., Sano, S., Yanase, T., Ohta, T. y Koyama, M. (2019). Optuna: A next-generation hyperparameter optimization framework. *Proceedings of the 25th ACM SIGKDD*, 2623–2631.
- Alfonzo, J. P., Iyer, K. G., Akiyama, M., et al. (2024). Decoding the imprints of past star formation on present-day morphology in galaxies with interpretable CNNs. *The Astrophysical Journal*, 967, 152.
- Dosovitskiy, A., Beyer, L., Kolesnikov, A., et al. (2021). An image is worth 16×16 words: Transformers for image recognition at scale. *ICLR*.
- Hu, E. J., Shen, Y., Wallis, P., et al. (2022). LoRA: Low-rank adaptation of large language models. *ICLR*.
- Lupton, R., Blanton, M. R., Fekete, G., et al. (2004). Preparing red-green-blue images from CCD data. *Publications of the Astronomical Society of the Pacific*, 116, 133.
- Parker, L., Lanusse, F., Shen, J., et al. (2025). AION-1: Omnimodal foundation model for astronomical sciences. *NeurIPS*. arXiv:2510.17960.
- Siméoni, O., Vo, H. V., Seitzer, M., et al. (2025). DINOv3. arXiv:2508.10104.
- Studer, S., Bui, T. B., Drescher, C., et al. (2021). Towards CRISP-ML(Q): A machine learning process model with quality assurance methodology. *Machine Learning and Knowledge Extraction*, 3, 392–413.
- UniverseTBD (2026). The Platonic Universe: Do foundation models see the same sky? arXiv:2509.19453v2.